In [ ]:
! pip install ripser

In [9]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ripser import ripser
from scipy.signal import butter, filtfilt
from scipy.stats import iqr, skew, kurtosis, entropy
from scipy.spatial.distance import pdist


# =========================
# CONFIG
# =========================
PKL_PATH = r"/content/drive/MyDrive/Colab Notebooks/BP/data/WESAD/S10_E4_Data/S10.pkl"
SUBJECT_NAME = "S10"

OUTPUT_DIR = r"/content/drive/MyDrive/Colab Notebooks/BP/result_wesad_tpv_s10"

FS_LABEL = 700
FS_BVP   = 64

WINDOW_SEC = 60
STEP_SEC   = 60   # non-overlap

USE_LABELS   = {1, 2}   # baseline, stress
STRESS_ID    = 2
MAJ_RATIO_TH = 0.95

LOWCUT = 0.5
HIGHCUT = 8.0
FILTER_ORDER = 4

SAVE_INDIVIDUAL_BOXPLOTS = True
SAVE_GRID_BOXPLOT = True

os.makedirs(OUTPUT_DIR, exist_ok=True)


# =========================
# 0) Basic preprocessing
# =========================
def fill_nan_with_median(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32).reshape(-1).copy()
    if np.isnan(x).any():
        med = np.nanmedian(x)
        if np.isnan(med):
            med = 0.0
        x = np.nan_to_num(x, nan=med, posinf=med, neginf=med)
    return x


def bandpass_filter(sig, fs=64, low=0.5, high=8.0, order=4):
    sig = np.asarray(sig, dtype=np.float32).reshape(-1)
    nyq = 0.5 * fs
    low_n = low / nyq
    high_n = high / nyq

    if not (0 < low_n < high_n < 1):
        raise ValueError(f"Invalid bandpass range: low={low}, high={high}, fs={fs}")

    b, a = butter(order, [low_n, high_n], btype="band")
    return filtfilt(b, a, sig).astype(np.float32)


# =========================
# 1) TPV (39-d)
# =========================
def extract_tpv_39(sig_1d: np.ndarray) -> np.ndarray:
    sig = np.asarray(sig_1d, dtype=np.float32).reshape(-1)
    if len(sig) < 3:
        return np.zeros(39, dtype=np.float32)

    s = float(np.std(sig))
    if s < 1e-8:
        return np.zeros(39, dtype=np.float32)
    sig = (sig - float(np.mean(sig))) / (s + 1e-8)

    X = np.stack([sig[:-1], sig[1:]], axis=1)

    dgms = ripser(X, maxdim=1)["dgms"]
    H1 = dgms[1]
    if H1.size == 0:
        return np.zeros(39, dtype=np.float32)

    births = H1[:, 0]
    deaths = H1[:, 1]
    lifetimes = deaths - births

    mask = np.isfinite(births) & np.isfinite(deaths) & np.isfinite(lifetimes)
    births, deaths, lifetimes = births[mask], deaths[mask], lifetimes[mask]
    if len(lifetimes) == 0:
        return np.zeros(39, dtype=np.float32)

    lifetimes_sorted = np.sort(lifetimes)

    feats = [
        float(np.mean(births)), float(np.std(births)),
        float(np.mean(deaths)), float(np.std(deaths)),
        float(np.mean(lifetimes)), float(np.std(lifetimes)),
        float(np.max(lifetimes)), float(np.min(lifetimes)),
        float(np.median(lifetimes)),
        float(iqr(lifetimes)),
        float(skew(lifetimes)) if len(lifetimes) > 2 else 0.0,
        float(kurtosis(lifetimes)) if len(lifetimes) > 3 else 0.0,

        float(np.max(births)), float(np.min(births)), float(np.median(births)),
        float(skew(births)) if len(births) > 2 else 0.0,
        float(kurtosis(births)) if len(births) > 3 else 0.0,

        float(np.max(deaths)), float(np.min(deaths)), float(np.median(deaths)),
        float(skew(deaths)) if len(deaths) > 2 else 0.0,
        float(kurtosis(deaths)) if len(deaths) > 3 else 0.0,

        float(len(lifetimes)),
        float(lifetimes_sorted[-1]),
        float(lifetimes_sorted[-2]) if len(lifetimes_sorted) > 1 else 0.0,
        float(np.max(lifetimes) / (np.min(lifetimes) + 1e-8)),
        float(np.sum(lifetimes ** 2)),
    ]

    lifetime_sum = float(np.sum(lifetimes)) + 1e-8
    lifetime_ratio = lifetimes / lifetime_sum
    PH_entropy = float(-np.sum(lifetime_ratio * np.log(lifetime_ratio + 1e-10)))

    bmin, bmax = float(np.min(births)), float(np.max(births))
    if bmax - bmin < 1e-8:
        Betti_entropy = 0.0
    else:
        hist, _ = np.histogram(births, bins=10, range=(bmin, bmax), density=True)
        Betti_entropy = float(entropy(hist + 1e-10))

    persistent_image_energy = float(np.sum(lifetimes ** 2))
    avg_persistence_distance = float(np.mean(pdist(lifetimes[:, None]))) if len(lifetimes) > 1 else 0.0

    if np.sum(lifetimes_sorted) > 0 and len(lifetimes_sorted) > 1:
        n = len(lifetimes_sorted)
        gini = (2*np.sum(np.arange(1, n+1)*lifetimes_sorted))/(n*np.sum(lifetimes_sorted)) - (n+1)/n
        Gini_index = float(gini)
    else:
        Gini_index = 0.0

    lifetime_variance = float(np.var(lifetimes))

    feats.extend([
        PH_entropy, Betti_entropy, persistent_image_energy,
        avg_persistence_distance, Gini_index, lifetime_variance
    ])

    feats = np.asarray(feats, dtype=np.float32)
    if feats.shape[0] < 39:
        feats = np.pad(feats, (0, 39 - feats.shape[0]), constant_values=0.0)
    elif feats.shape[0] > 39:
        feats = feats[:39]

    return feats


# =========================
# 2) Build TPV table
# =========================
def build_bvp_tpv_table(
    pkl_path: str,
    subject_name: str,
    window_sec: int,
    step_sec: int,
    fs_label: int = 700,
    fs_bvp: int = 64,
    use_labels: set = {1, 2},
    stress_id: int = 2,
    maj_ratio_th: float = 0.95,
    verbose: bool = True,
) -> pd.DataFrame:

    with open(pkl_path, "rb") as f:
        s = pickle.load(f, encoding="latin1")

    bvp = np.asarray(s["signal"]["wrist"]["BVP"]).reshape(-1).astype(np.float32)
    labels = np.asarray(s["label"]).reshape(-1).astype(np.int64)

    print(f"[DEBUG] raw bvp len   : {len(bvp)}")
    print(f"[DEBUG] raw label len : {len(labels)}")

    dur_wrist = len(bvp) / fs_bvp
    labels = labels[: int(dur_wrist * fs_label)]

    print(f"[DEBUG] aligned label len : {len(labels)}")
    print(f"[DEBUG] unique labels      : {np.unique(labels)}")

    bvp = fill_nan_with_median(bvp)
    bvp = bandpass_filter(bvp, fs=fs_bvp, low=LOWCUT, high=HIGHCUT, order=FILTER_ORDER)

    Wl = int(window_sec * fs_label)
    Sl = int(step_sec   * fs_label)
    Wb = int(window_sec * fs_bvp)

    print(f"[DEBUG] Wl={Wl}, Sl={Sl}, Wb={Wb}")

    tpv_rows = []
    cnt_total = 0
    cnt_kept = 0

    for start_l in range(0, len(labels) - Wl + 1, Sl):
        end_l = start_l + Wl
        win_lab = labels[start_l:end_l]
        cnt_total += 1

        binc = np.bincount(win_lab)
        maj = int(np.argmax(binc))
        maj_ratio = float((win_lab == maj).mean())

        if maj not in use_labels:
            continue
        if maj_ratio < maj_ratio_th:
            continue

        t0 = start_l / fs_label
        start_b = int(round(t0 * fs_bvp))
        end_b = start_b + Wb

        if end_b > len(bvp):
            continue

        seg_bvp = bvp[start_b:end_b]
        if len(seg_bvp) != Wb:
            continue

        tpv = extract_tpv_39(seg_bvp)

        row = {
            "subject": subject_name,
            "pkl_path": pkl_path,
            "t_start_sec": float(t0),
            "t_end_sec": float(t0 + window_sec),
            "label_major": maj,
            "label_name": "stress" if maj == stress_id else "normal",
            "label_bin": 1 if maj == stress_id else 0,
            "major_ratio": maj_ratio,
        }
        row.update({f"BVP_TPV_{j}": float(tpv[j]) for j in range(39)})
        tpv_rows.append(row)

        cnt_kept += 1
        if verbose:
            print(f"[KEEP {cnt_kept}] t0={t0:.1f}s label_major={maj} ratio={maj_ratio:.3f}")

    print(f"[DEBUG] total windows checked = {cnt_total}")
    print(f"[DEBUG] total windows kept    = {cnt_kept}")

    return pd.DataFrame(tpv_rows)


# =========================
# 3) Summary table
# =========================
def build_summary_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    df_n = df[df["label_bin"] == 0]
    df_s = df[df["label_bin"] == 1]

    for i in range(39):
        col = f"BVP_TPV_{i}"
        rows.append({
            "feature": col,
            "normal_mean": df_n[col].mean(),
            "normal_std": df_n[col].std(),
            "normal_median": df_n[col].median(),
            "stress_mean": df_s[col].mean(),
            "stress_std": df_s[col].std(),
            "stress_median": df_s[col].median(),
            "mean_diff(stress-normal)": df_s[col].mean() - df_n[col].mean(),
            "median_diff(stress-normal)": df_s[col].median() - df_n[col].median(),
        })

    return pd.DataFrame(rows)


# =========================
# 4) Boxplots
# =========================
def plot_individual_boxplots(df: pd.DataFrame, save_dir: str):
    os.makedirs(save_dir, exist_ok=True)

    for i in range(39):
        col = f"BVP_TPV_{i}"
        plt.figure(figsize=(5.2, 4.0))
        sns.boxplot(data=df, x="label_name", y=col, order=["normal", "stress"])
        plt.title(f"{col}: stress vs normal")
        plt.xlabel("")
        plt.ylabel(col)
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, f"{col}_boxplot.png"), dpi=200)
        plt.close()


def plot_grid_boxplots(df: pd.DataFrame, save_path: str, n_cols: int = 4):
    n_feats = 39
    n_rows = int(np.ceil(n_feats / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.3*n_cols, 3.4*n_rows))
    axes = np.array(axes).reshape(-1)

    for i in range(n_feats):
        ax = axes[i]
        col = f"BVP_TPV_{i}"
        sns.boxplot(data=df, x="label_name", y=col, order=["normal", "stress"], ax=ax)
        ax.set_title(col, fontsize=9)
        ax.set_xlabel("")
        ax.set_ylabel("")

    for j in range(n_feats, len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()


# =========================
# RUN
# =========================
if __name__ == "__main__":
    df_all = build_bvp_tpv_table(
        pkl_path=PKL_PATH,
        subject_name=SUBJECT_NAME,
        window_sec=WINDOW_SEC,
        step_sec=STEP_SEC,
        fs_label=FS_LABEL,
        fs_bvp=FS_BVP,
        use_labels=USE_LABELS,
        stress_id=STRESS_ID,
        maj_ratio_th=MAJ_RATIO_TH,
        verbose=True,
    )

    print("\n[INFO] final shape:", df_all.shape)

    if len(df_all) == 0:
        print("\n[WARN] No valid windows were extracted.")
        print("Try lowering MAJ_RATIO_TH from 0.95 to 0.90 or 0.80.")
    else:
        print(df_all[["subject", "t_start_sec", "label_major", "label_name", "major_ratio"]].head())

        main_csv = os.path.join(OUTPUT_DIR, f"{SUBJECT_NAME}_wesad_wrist_bvp_tpv39_baseline_vs_stress.csv")
        df_all.to_csv(main_csv, index=False)
        print(f"[INFO] Saved main CSV: {main_csv}")

        summary_df = build_summary_table(df_all)
        summary_csv = os.path.join(OUTPUT_DIR, f"{SUBJECT_NAME}_wesad_wrist_bvp_tpv39_summary.csv")
        summary_df.to_csv(summary_csv, index=False)
        print(f"[INFO] Saved summary CSV: {summary_csv}")

        print("\n[INFO] Label counts")
        print(df_all["label_name"].value_counts())

        if SAVE_INDIVIDUAL_BOXPLOTS:
            indiv_dir = os.path.join(OUTPUT_DIR, "boxplots")
            plot_individual_boxplots(df_all, indiv_dir)
            print(f"[INFO] Saved individual boxplots -> {indiv_dir}")

        if SAVE_GRID_BOXPLOT:
            grid_path = os.path.join(OUTPUT_DIR, f"{SUBJECT_NAME}_tpv39_boxplot_grid.png")
            plot_grid_boxplots(df_all, grid_path, n_cols=4)
            print(f"[INFO] Saved grid boxplot -> {grid_path}")

[DEBUG] raw bvp len   : 351744
[DEBUG] raw label len : 3847200
[DEBUG] aligned label len : 3847200
[DEBUG] unique labels      : [0 1 2 3 4 5 6 7]
[DEBUG] Wl=42000, Sl=42000, Wb=3840
[KEEP 1] t0=120.0s label_major=1 ratio=1.000
[KEEP 2] t0=180.0s label_major=1 ratio=1.000
[KEEP 3] t0=240.0s label_major=1 ratio=1.000
[KEEP 4] t0=300.0s label_major=1 ratio=1.000
[KEEP 5] t0=360.0s label_major=1 ratio=1.000
[KEEP 6] t0=420.0s label_major=1 ratio=1.000
[KEEP 7] t0=480.0s label_major=1 ratio=1.000
[KEEP 8] t0=540.0s label_major=1 ratio=1.000
[KEEP 9] t0=600.0s label_major=1 ratio=1.000
[KEEP 10] t0=660.0s label_major=1 ratio=1.000
[KEEP 11] t0=720.0s label_major=1 ratio=1.000
[KEEP 12] t0=780.0s label_major=1 ratio=1.000
[KEEP 13] t0=840.0s label_major=1 ratio=1.000
[KEEP 14] t0=900.0s label_major=1 ratio=1.000
[KEEP 15] t0=960.0s label_major=1 ratio=1.000
[KEEP 16] t0=1020.0s label_major=1 ratio=1.000
[KEEP 17] t0=1080.0s label_major=1 ratio=1.000
[KEEP 18] t0=1140.0s label_major=1 ratio=1.